# 02 — One-Click Weight-Space Mixer & Live `bim-mcp` Lab

Pull community-trained **MutinyBIM Trade LoRAs** (`google/gemma-4-31B-it` weight space) from Hugging Face, compose them mathematically via **Task Arithmetic**, **Clash-Vector Negation** (`-λ·ΔW_clash`), **TIES**, **DARE**, or **CMA-ES EvoMerge** against `bim-lint`, inspect **60-layer SVD diagnostics**, and serve the merged brain straight into `bim-mcp` / `bim-editor`!

### Hardware & Credit Budget
* **CPU:** Try the synthetic merge first. A 24 GB L4 is a candidate for testing 4-bit inference with real adapters; fit and cost are unverified.
* **A100 (40GB/80GB):** Optional if you want faster EvoMerge iterations or bf16 merged serving.


In [ ]:
#@title 1. Clone Repos, Install Node 22, Start Warm Verifier, Install Python Deps
!nvidia-smi

import json, subprocess, sys, time
from pathlib import Path

MIXER_REPO_URL = "https://github.com/5hyycjb77y/mutinybim-mixer.git" #@param {type:"string"}
WORK_DIR = Path("/content/work")
WORK_DIR.mkdir(parents=True, exist_ok=True)
%cd /content/work

if not (WORK_DIR / "mutinybim-mixer").exists():
    !git clone --depth 1 {MIXER_REPO_URL} mutinybim-mixer || true

MIXER_DIR = WORK_DIR / "mutinybim-mixer"
assert MIXER_DIR.exists(), "Upload mutinybim-mixer to /content/work/mutinybim-mixer or set MIXER_REPO_URL."

SPEC = json.loads((MIXER_DIR / "spec.json").read_text())
REGISTRY = json.loads((MIXER_DIR / "registry.json").read_text())
MUTINY_DIR = WORK_DIR / "mutinybim"

if not MUTINY_DIR.exists():
    !git clone {SPEC["mutinybim"]["repo"]} {MUTINY_DIR}
!git -C {MUTINY_DIR} fetch --depth 1 origin {SPEC["mutinybim"]["commit"]} && git -C {MUTINY_DIR} checkout {SPEC["mutinybim"]["commit"]}

node_v = subprocess.run(["node", "-v"], capture_output=True, text=True)
if node_v.returncode != 0 or int(node_v.stdout.strip().lstrip("v").split(".")[0]) < 22:
    !curl -fsSL https://deb.nodesource.com/setup_22.x | bash - && apt-get install -y nodejs

if not (MUTINY_DIR / "node_modules").exists():
    !npm --prefix {MUTINY_DIR} install --no-audit --no-fund

verifier_log = open("/content/verifier.log", "w")
verifier_proc = subprocess.Popen(
    ["node", str(MIXER_DIR / "verifier/server.mjs"), "--mutinybim", str(MUTINY_DIR), "--port", "8787"],
    stdout=verifier_log,
    stderr=subprocess.STDOUT,
)

!pip install -q torch safetensors huggingface_hub cma matplotlib peft unsloth fastapi uvicorn
sys.path.insert(0, str(MIXER_DIR))

from verifier.verify_client import MutinyVerifierClient
client = MutinyVerifierClient("http://127.0.0.1:8787")
for _ in range(30):
    try:
        print("Verifier ready:", client.health())
        break
    except Exception:
        time.sleep(1)


In [ ]:
#@title 2. Pull Community Trade LoRAs (or Generate Demo Adapters to Test Plumbing)
import torch
from safetensors.torch import save_file
from mixer.lora_ops import check_adapter_compatibility, load_lora_deltas

# Override or add HF repo IDs / local paths here, or leave blank to use registry.json
ADAPTER_SPECS_JSON = "[]" #@param {type:"string"}
DEMO_SYNTHETIC_FALLBACK = True #@param {type:"boolean"}

specs_to_load = json.loads(ADAPTER_SPECS_JSON)
if not specs_to_load:
    specs_to_load = REGISTRY.get("adapters", [])

loaded_adapters = []  # list of dicts: {id, trade, weight, deltas, config}

if specs_to_load:
    for entry in specs_to_load:
        deltas, cfg = load_lora_deltas(entry["hf_repo"], revision=entry.get("revision"))
        issues = check_adapter_compatibility(cfg, SPEC)
        if issues:
            print(f"[SKIP] {entry['id']} incompatible:", issues)
            continue
        loaded_adapters.append({
            "id": entry["id"],
            "trade": entry["trade"],
            "weight": float(entry.get("default_weight", 0.8)),
            "deltas": deltas,
            "config": cfg,
        })
        print(f"[OK] Loaded {entry['id']} ({entry['trade']}) — {len(deltas)} delta matrices")

elif DEMO_SYNTHETIC_FALLBACK:
    print("No published adapters in registry.json yet — generating tiny synthetic rank-16 demo LoRAs to verify weight-space math.")
    demo_dir = Path("/content/demo_adapters")
    demo_trades = [
        ("demo-arch", "architecture", 0.8, 101),
        ("demo-mech", "mechanical", 0.7, 202),
        ("demo-clash", "clash_negation", -0.3, 303),
    ]
    for name, trade, def_w, seed in demo_trades:
        p = demo_dir / name
        p.mkdir(parents=True, exist_ok=True)
        cfg = {
            "base_model_name_or_path": SPEC["base_model"]["hf_id"],
            "r": SPEC["lora"]["r"],
            "lora_alpha": SPEC["lora"]["lora_alpha"],
            "bias": "none",
            "target_modules": SPEC["lora"]["target_modules"],
            "peft_type": "LORA",
            "task_type": "CAUSAL_LM",
        }
        (p / "adapter_config.json").write_text(json.dumps(cfg, indent=2))
        g = torch.Generator().manual_seed(seed)
        tensors = {}
        for layer_idx in range(4):
            for mod in ("q_proj", "v_proj", "gate_proj"):
                prefix = f"base_model.model.model.language_model.layers.{layer_idx}.self_attn.{mod}" if mod in ("q_proj", "v_proj") else f"base_model.model.model.language_model.layers.{layer_idx}.mlp.{mod}"
                tensors[f"{prefix}.lora_A.weight"] = torch.randn(16, 64, generator=g) * 0.01
                tensors[f"{prefix}.lora_B.weight"] = torch.randn(64, 16, generator=g) * 0.01
        save_file(tensors, str(p / "adapter_model.safetensors"))
        deltas, cfg = load_lora_deltas(p)
        loaded_adapters.append({"id": name, "trade": trade, "weight": def_w, "deltas": deltas, "config": cfg})
        print(f"[DEMO] Created {name} ({trade})")

assert loaded_adapters, "No adapters loaded!" 


In [ ]:
#@title 3. Interactive Mixing Desk (Task Arithmetic / TIES / DARE + Clash Negation)
from mixer.lora_ops import merge_deltas, refactorize_deltas_to_lora, save_merged_lora

MERGE_METHOD = "dare_ties" #@param ["linear", "ties", "dare_linear", "dare_ties"]
TIES_DENSITY = 0.5 #@param {type:"slider", min:0.1, max:1.0, step:0.05}
DARE_DROP_RATE = 0.2 #@param {type:"slider", min:0.0, max:0.8, step:0.05}
CLASH_NEGATION_WEIGHT = -0.3 #@param {type:"slider", min:-1.2, max:0.0, step:0.05}
OUT_RANK = 16 #@param {type:"integer"}

weights = []
for a in loaded_adapters:
    w = CLASH_NEGATION_WEIGHT if a["trade"] == "clash_negation" else a["weight"]
    weights.append(w)
    print(f"  {a['id']} ({a['trade']}): weight={w:+.2f}")

merged_deltas = merge_deltas(
    [a["deltas"] for a in loaded_adapters],
    weights,
    method=MERGE_METHOD,
    density=TIES_DENSITY,
    drop_rate=DARE_DROP_RATE,
)
lora_tensors, svd_stats = refactorize_deltas_to_lora(
    merged_deltas,
    out_rank=OUT_RANK,
    lora_alpha=OUT_RANK,
)
MERGED_DIR = save_merged_lora(
    lora_tensors,
    loaded_adapters[0]["config"],
    "/content/outputs/merged_lora",
    out_rank=OUT_RANK,
    lora_alpha=OUT_RANK,
    metadata={"method": MERGE_METHOD, "weights": weights, "svd_stats": svd_stats},
)
print("Merged LoRA saved to:", MERGED_DIR)
print("SVD refactorization stats:", json.dumps(svd_stats, indent=2))


In [ ]:
#@title 4. Weight-Space SVD & Trade-Orthogonality Diagnostics (60 Layers)
import matplotlib.pyplot as plt
from mixer.lora_ops import svd_diagnostics

if len(loaded_adapters) >= 2:
    diag = svd_diagnostics(loaded_adapters[0]["deltas"], loaded_adapters[1]["deltas"], top_k=4)
    print(
        f"Global mean cosine similarity ({loaded_adapters[0]['trade']} vs {loaded_adapters[1]['trade']}):",
        diag["global_mean_cosine_similarity"],
    )
    layers = sorted(k for k in diag["layers"].keys() if k >= 0)
    frobs = [diag["layers"][k]["mean_frobenius_norm"] for k in layers]
    ratios = [diag["layers"][k]["mean_top_k_energy_ratio"] for k in layers]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(layers, frobs, marker="o")
    axes[0].set_title(f"{loaded_adapters[0]['trade']} Delta Frobenius Norm by Layer")
    axes[0].set_xlabel("Gemma 4 Layer")
    axes[0].set_ylabel("Mean ||ΔW||_F")

    axes[1].plot(layers, ratios, marker="s", color="tab:orange")
    axes[1].set_title("Top-4 Singular Energy Ratio by Layer")
    axes[1].set_xlabel("Gemma 4 Layer")
    axes[1].set_ylabel("Energy Ratio")
    plt.tight_layout()
    plt.show()
else:
    print("Load at least 2 adapters to compare pairwise trade orthogonality.")


In [ ]:
#@title 5. Evaluate Merged Brain on Held-Out Multi-Trade Benchmark

LOAD_MODEL_FOR_EVAL = False #@param {type:"boolean"}

if LOAD_MODEL_FOR_EVAL:
    if not specs_to_load:
        raise RuntimeError("Model evaluation requires real compatible adapters; synthetic adapters are for merge checks only.")
    from unsloth import FastModel
    from peft import PeftModel
    from mixer.prompts import MULTI_TRADE_BENCHMARK, format_task_prompt
    from verifier.verify_client import extract_typescript
    ref_md = client.reference(compact=True)["markdown"]
    base_model, tokenizer = FastModel.from_pretrained(
        model_name=SPEC["base_model"]["unsloth_id"],
        max_seq_length=4096,
        dtype=None,
        load_in_4bit=True,
    )
    eval_model = PeftModel.from_pretrained(base_model, str(MERGED_DIR))
    FastModel.for_inference(eval_model)

    def evaluate_current_model(tasks):
        rows = []
        for task in tasks:
            msgs = format_task_prompt(task, ref_md[:4000])
            try:
                prompt_text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=True)
            except TypeError:
                prompt_text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            inputs = tokenizer(prompt_text, return_tensors="pt").to(eval_model.device)
            with torch.no_grad():
                out_ids = eval_model.generate(**inputs, max_new_tokens=1536, temperature=0.6, top_p=0.95, top_k=64, do_sample=True)
            comp = tokenizer.decode(out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
            ts = extract_typescript(comp)
            vr = client.verify(ts, start=task.get("start", "empty"), expected_kinds=task.get("expected_kinds"), min_elements=task.get("min_elements"))
            rows.append({"id": task["id"], "ok": vr.ok, "reward": vr.reward, "clashes": vr.lint_summary.get("errors", 0)})
        return rows

    results = evaluate_current_model(MULTI_TRADE_BENCHMARK)
    mean_reward = sum(r["reward"] for r in results) / len(results)
    print(f"Multi-trade mean reward: {mean_reward:.3f}")
    for r in results:
        print(" ", r)


In [ ]:
#@title 6. CMA-ES EvoMerge Against bim-lint (Global or 6 Depth Bands)
from mixer.evomerge import EvoMergeConfig, EvoMergeOptimizer

RUN_EVOMERGE = False #@param {type:"boolean"}
EVO_MODE = "depth_bands" #@param ["global", "depth_bands"]
EVO_GENERATIONS = 6 #@param {type:"integer"}
EVO_POP_SIZE = 6 #@param {type:"integer"}

if RUN_EVOMERGE and LOAD_MODEL_FOR_EVAL:
    clash_indices = tuple(i for i, a in enumerate(loaded_adapters) if a["trade"] == "clash_negation")
    evo_cfg = EvoMergeConfig(
        mode=EVO_MODE,
        num_bands=6,
        num_layers=SPEC["base_model"]["num_hidden_layers"],
        method=MERGE_METHOD,
        population_size=EVO_POP_SIZE,
        generations=EVO_GENERATIONS,
        allow_negative_indices=clash_indices,
    )
    opt = EvoMergeOptimizer(len(loaded_adapters), evo_cfg)
    fast_bench = MULTI_TRADE_BENCHMARK[:3]

    def evo_eval(genome):
        cand_deltas = merge_deltas(
            [a["deltas"] for a in loaded_adapters],
            genome["weights"],
            method=genome["method"],
            density=genome["density"],
            drop_rate=DARE_DROP_RATE,
            layer_schedules=genome["layer_schedules"],
        )
        cand_lora, _ = refactorize_deltas_to_lora(cand_deltas, out_rank=OUT_RANK, lora_alpha=OUT_RANK, use_lowrank=True)
        cand_dir = save_merged_lora(cand_lora, loaded_adapters[0]["config"], "/content/outputs/evo_cand", out_rank=OUT_RANK, lora_alpha=OUT_RANK)
        eval_model.load_adapter(str(cand_dir), adapter_name="evo")
        eval_model.set_adapter("evo")
        scores = evaluate_current_model(fast_bench)
        eval_model.delete_adapter("evo")
        return sum(s["reward"] for s in scores) / len(scores)

    evo_res = opt.optimize(evo_eval, init_weights=weights)
    print(f"EvoMerge ({evo_res.backend}) best fitness: {evo_res.best_fitness:.4f}")
    print("Best genome:", evo_res.best_genome)
    # Save winning genome as final merged LoRA
    win_deltas = merge_deltas(
        [a["deltas"] for a in loaded_adapters],
        evo_res.best_genome["weights"],
        method=evo_res.best_genome["method"],
        density=evo_res.best_genome["density"],
        drop_rate=DARE_DROP_RATE,
        layer_schedules=evo_res.best_genome["layer_schedules"],
    )
    win_lora, _ = refactorize_deltas_to_lora(win_deltas, out_rank=OUT_RANK, lora_alpha=OUT_RANK)
    MERGED_DIR = save_merged_lora(win_lora, loaded_adapters[0]["config"], "/content/outputs/merged_lora_evo", out_rank=OUT_RANK, lora_alpha=OUT_RANK)


In [ ]:
#@title 7. Serve Merged Gemma 4 Brain to Local MutinyBIM `bim-mcp`
SERVE_LIVE_ENDPOINT = False #@param {type:"boolean"}

if SERVE_LIVE_ENDPOINT and LOAD_MODEL_FOR_EVAL:
    # Serve directly via FastAPI + Unsloth so Gemma 4 runs cleanly without vLLM dynamic-LoRA caveats
    import threading, uvicorn
    from fastapi import FastAPI
    from pydantic import BaseModel

    app = FastAPI()

    class ChatReq(BaseModel):
        model: str = "mutinybim-gemma4-merged"
        messages: list[dict]
        temperature: float = 0.7
        max_tokens: int = 2048

    @app.post("/v1/chat/completions")
    def chat_completions(req: ChatReq):
        try:
            prompt_text = tokenizer.apply_chat_template(req.messages, tokenize=False, add_generation_prompt=True, enable_thinking=True)
        except TypeError:
            prompt_text = tokenizer.apply_chat_template(req.messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt_text, return_tensors="pt").to(eval_model.device)
        with torch.no_grad():
            out_ids = eval_model.generate(**inputs, max_new_tokens=req.max_tokens, temperature=req.temperature, top_p=0.95, top_k=64, do_sample=True)
        text = tokenizer.decode(out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
        return {"choices": [{"message": {"role": "assistant", "content": text}}]}

    threading.Thread(target=lambda: uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning"), daemon=True).start()
    print("Loopback only: http://127.0.0.1:8000/v1. Run the bridge in the same runtime, or host inference on a machine you control.")